# Machine Learning for CICY 4-Folds

H. Erbin, R. Finotello

Following the analysis in [arXiv:2007.13379](https://arxiv.org/abs/2007.13379) and [arXiv:2007.15706](https://arxiv.org/abs/2007.15706), we apply similar concepts to CICY 4-folds.
The idea is to see whether the Inception network can also be applied in higher dimensions.

## Feature Engineering

In this notebook we first build some engineered features providing additional information with respect to the original dataset.

In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import os

## Download and Read the Dataset

The dataset was first introduced in [arXiv:1405.2073](http://arxiv.org/abs/1405.2073).
The authors provide Hodge numbers and other characteristic invariants of CICY 4-folds.

In [2]:
df = pd.read_json('https://www.lpthe.jussieu.fr/~erbin/files/data/cicy4.json.gz', orient='index')

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 921497 entries, 1 to 921497
Data columns (total 9 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   euler   921497 non-null  int64  
 1   favour  921497 non-null  bool   
 2   h11     905684 non-null  float64
 3   h21     905684 non-null  float64
 4   h22     905684 non-null  float64
 5   h31     905684 non-null  float64
 6   isprod  921497 non-null  bool   
 7   matrix  921497 non-null  object 
 8   size    921497 non-null  object 
dtypes: bool(2), float64(4), int64(1), object(2)
memory usage: 58.0+ MB


## Feature Engineering

Notice that each configuration matrix is a $m \times k$ table
\begin{equation}
    X =
    \begin{bmatrix}
        \mathbb{P}^{n_1}\colon & a_1^1 & \dots & a_k^1 \\
        \mathbb{P}^{n_2}\colon & a_1^2 & \dots & a_k^2 \\
        \vdots & & & \\
        \mathbb{P}^{n_m}\colon & a_1^m & \dots & a_k^m \\
    \end{bmatrix}
\end{equation}
and
\begin{equation}
    n_r = \sum\limits_{\alpha = 1}^k\, a_{\alpha}^r - 1,
    \qquad
    r = 1, 2, \dots, m.
\end{equation}

We then compute the number of projective spaces $m$ (i.e. the number of rows), the number of equations $k$ (i.e. the number of columns), the number of $\mathbb{P}^1$ $f$ (i.e. the number of rows such that $n_r = 1$ $\forall r = 1, 2, \dots, m$), the number of $\mathbb{P}^2$ (i.e. the number of rows such that $n_r = 2$ $\forall r = 1, 2, \dots, m$), and the number $F$ of $\mathbb{P}^{n_r}$ such that $n_r \neq 1$.

The excess number is then computed as
\begin{equation}
    N_{ex} = \sum\limits_{r = 1}^F\, n_r + f + m - 2 k.
\end{equation}

The norm of the matrix is the Frobenius norm $\left|\left| A \right|\right| = \sqrt{\sum\limits_{r = 1}^m \sum\limits_{\alpha = 1}^k\, \left| a_{\alpha}^r \right|^2}$.

Other engineered features include the list of the dimensions of the projective spaces and the list of the degrees of the polynomials.

In [4]:
# no. of projective spaces (rows)
df['num_cp']         = df['size'].apply(lambda s: s[0]).astype(np.int)

# no. of equations (columns)
df['num_eqs']        = df['size'].apply(lambda s: s[1]).astype(np.int)

# no. of P^1
df['num_cp_1']       = df['matrix'].apply(lambda m: np.sum(m, axis=1) - 1).apply(lambda v: np.sum((v == 1).astype(np.int)))

# no. of P^2
df['num_cp_2']       = df['matrix'].apply(lambda m: np.sum(m, axis=1) - 1).apply(lambda v: np.sum((v == 2).astype(np.int)))

# no. of P^n with n != 1
df['num_cp_neq1']    = df['matrix'].apply(lambda m: np.sum(m, axis=1) - 1).apply(lambda v: np.sum((v > 1).astype(np.int)))

# excess number
df['num_ex']         = df['matrix'].apply(lambda m: np.sum(m, axis=1) - 1).apply(lambda v: np.sum(v[(v > 1)])) \
                       + df['num_cp_1'] + df['num_cp'] - 2 * df['num_eqs']

# Frobenius norm and rank of the matrix
df['norm_matrix']    = df['matrix'].apply(np.linalg.norm)
df['rank_matrix']    = df['matrix'].apply(np.linalg.matrix_rank)

# list of the dimensions of the projective spaces
df['dim_cp']         = df['matrix'].apply(lambda m: np.sum(m, axis=1) - 1)
df['min_dim_cp']     = df['dim_cp'].apply(np.min)
df['max_dim_cp']     = df['dim_cp'].apply(np.max)
df['mean_dim_cp']    = df['dim_cp'].apply(np.mean)
df['std_dim_cp']     = df['dim_cp'].apply(np.std)
df['median_dim_cp']  = df['dim_cp'].apply(np.median)
df['quart_dim_cp']   = df['dim_cp'].apply(lambda l: np.quantile(l, [0.25, 0.75]))

# list of the degrees of the equations
df['deg_eqs']        = df['matrix'].apply(lambda m: np.max(m, axis=1))
df['min_deg_eqs']    = df['deg_eqs'].apply(np.min)
df['max_deg_eqs']    = df['deg_eqs'].apply(np.max)
df['mean_deg_eqs']   = df['deg_eqs'].apply(np.mean)
df['std_deg_eqs']    = df['deg_eqs'].apply(np.std)
df['median_deg_eqs'] = df['deg_eqs'].apply(np.median)
df['quart_deg_eqs']  = df['deg_eqs'].apply(lambda l: np.quantile(l, [0.25, 0.75]))

## Save the Dataset

We then save the dataset to file.

In [5]:
# create the directory
os.makedirs('./data', exist_ok=True)

# write the file to file
df.to_json('./data/cicy4_eng.json.gz', orient='index')
df.to_csv('./data/cicy4_eng.csv.gz', index=False)